# Manual: embeddings, chunking, busqueda semantica y RAG

Este notebook muestra el flujo completo sin framework. La idea es ver que hace cada parte por dentro antes de usar LangChain o LlamaIndex.

Flujo de la clase:

| Paso | Que hacemos | Para que sirve |
|---|---|---|
| 1 | Cargamos documentos | Tener una mini base de conocimiento |
| 2 | Dividimos en chunks | Buscar fragmentos pequenos, no documentos enteros |
| 3 | Creamos embeddings | Convertir texto en vectores numericos |
| 4 | Calculamos similitud coseno | Medir que chunk se parece mas a la pregunta |
| 5 | Armamos Top-K | Quedarnos con los mejores resultados |
| 6 | Enviamos contexto al LLM | Responder usando evidencia recuperada |

In [ ]:
import sys
from pathlib import Path

import numpy as np
from dotenv import load_dotenv
from openai import OpenAI

BASE = Path.cwd()
if not (BASE / 'data' / 'documentos.py').exists(): BASE = BASE.parent
if not (BASE / 'data' / 'documentos.py').exists(): BASE = Path('ai_engineer/ejercicios_embeddings_simple').resolve()
load_dotenv(BASE.parent / '.env'); sys.path.append(str(BASE / 'data'))

from documentos import DOCUMENTOS, PREGUNTA

client = OpenAI()

## 1. Documentos

Un documento es el texto original. En un sistema real podria ser un PDF, una pagina de ayuda, una politica interna o un ticket.

Aca usamos tres documentos muy chicos para que sea facil seguir el ejemplo:

| Documento | Tema |
|---|---|
| `manual_red.md` | Problemas de conexion e Internet |
| `facturacion.md` | Facturas y cobros duplicados |
| `seguridad.md` | Recuperacion de contrasena |

In [ ]:
# Mostramos cada documento tal cual esta guardado.
for doc in DOCUMENTOS:
    print(f"--- {doc['source']} ---")
    print(doc['text'].strip(), '\n')

print('Pregunta del usuario:', PREGUNTA)

## 2. Chunking manual

Un chunk es un fragmento pequeno del documento. No queremos buscar sobre documentos completos porque mezclan muchos temas.

Para esta primera clase usamos una estrategia simple: **cada linea no vacia es un chunk**.

Esto no es lo mas avanzado, pero es perfecto para entender la idea.

| Campo | Significado |
|---|---|
| `source` | De que documento salio el chunk |
| `text` | Texto que vamos a convertir en embedding |

In [ ]:
# Cada linea no vacia de cada documento se convierte en un chunk.
chunks = [
    {'source': doc['source'], 'text': linea.strip()}
    for doc in DOCUMENTOS
    for linea in doc['text'].splitlines()
    if linea.strip()
]

for i, chunk in enumerate(chunks, start=1):
    print(f"{i}. ({chunk['source']}) {chunk['text']}")

### Grafico: longitud de cada chunk

Este grafico ayuda a ver que no todos los fragmentos tienen el mismo tamano.

En RAG, chunks demasiado largos pueden traer ruido. Chunks demasiado cortos pueden perder contexto.

In [ ]:
import matplotlib.pyplot as plt

longitudes = [len(chunk['text'].split()) for chunk in chunks]

plt.figure(figsize=(8, 3))
plt.bar(range(1, len(chunks) + 1), longitudes)
plt.title('Cantidad de palabras por chunk')
plt.xlabel('Chunk')
plt.ylabel('Palabras')
plt.show()

## 3. Embeddings

Un embedding convierte texto en una lista de numeros. Esa lista representa el significado del texto en un espacio vectorial.

No interpretamos cada numero por separado. Lo importante es comparar vectores completos.

| Texto | Embedding |
|---|---|
| Pregunta | Vector de la pregunta |
| Chunk 1 | Vector del chunk 1 |
| Chunk 2 | Vector del chunk 2 |
| ... | ... |

Regla importante: la pregunta y los chunks deben usar **el mismo modelo de embeddings**.

In [ ]:
# Docs: https://platform.openai.com/docs/guides/embeddings
# API ref: https://platform.openai.com/docs/api-reference/embeddings
respuesta = client.embeddings.create(
    model='text-embedding-3-small',
    input=[chunk['text'] for chunk in chunks],
)
vectores = [np.array(item.embedding) for item in respuesta.data]

respuesta_pregunta = client.embeddings.create(
    model='text-embedding-3-small',
    input=PREGUNTA,
)
vector_pregunta = np.array(respuesta_pregunta.data[0].embedding)

print('Cantidad de chunks:', len(chunks))
print('Cantidad de embeddings:', len(vectores))
print('Dimensiones de cada embedding:', len(vectores[0]))

## 4. Similitud coseno

La similitud coseno mide si dos vectores apuntan en una direccion parecida.

Formula conceptual:

```text
coseno = producto_punto(a, b) / (norma(a) * norma(b))
```

Cuanto mas alto el score, mas parecido semanticamente es el chunk a la pregunta.

In [ ]:
# numpy.dot: https://numpy.org/doc/stable/reference/generated/numpy.dot.html
# numpy.linalg.norm: https://numpy.org/doc/stable/reference/generated/numpy.linalg.norm.html
scores = [
    np.dot(vector_pregunta, vector) / (np.linalg.norm(vector_pregunta) * np.linalg.norm(vector))
    for vector in vectores
]

top_3 = sorted(zip(chunks, scores), key=lambda item: item[1], reverse=True)[:3]

for chunk, score in top_3:
    print(f'[{score:.4f}] ({chunk["source"]}) {chunk["text"]}')

### Grafico: scores de similitud

Este grafico permite ver rapidamente que fragmentos quedaron mas cerca de la pregunta.

El chunk con score mas alto deberia hablar sobre WiFi o conexion, porque la pregunta es sobre reconectar el WiFi.

In [ ]:
plt.figure(figsize=(8, 3))
plt.bar(range(1, len(scores) + 1), scores)
plt.title('Similitud coseno entre pregunta y chunks')
plt.xlabel('Chunk')
plt.ylabel('Score')
plt.show()

## 5. Mini grafico 2D educativo

Los embeddings reales tienen muchas dimensiones. Para dibujar algo simple, tomamos solo las primeras dos dimensiones.

Este grafico no es una visualizacion perfecta del significado, pero ayuda a imaginar la idea: textos parecidos viven cerca en el espacio vectorial.

In [ ]:
plt.figure(figsize=(5, 5))
plt.scatter(vector_pregunta[0], vector_pregunta[1], label='pregunta', s=120)

for i, vector in enumerate(vectores, start=1):
    plt.scatter(vector[0], vector[1])
    plt.text(vector[0], vector[1], str(i))

plt.title('Vista 2D educativa de embeddings')
plt.xlabel('dimension 1')
plt.ylabel('dimension 2')
plt.legend()
plt.show()

## 6. RAG

RAG significa **Retrieval-Augmented Generation**.

Primero recuperamos evidencia con busqueda semantica. Despues se la damos al modelo para que responda.

| Parte | Que contiene |
|---|---|
| Pregunta | Lo que quiere saber el usuario |
| Contexto | Los chunks Top-K recuperados |
| LLM | Redacta la respuesta usando ese contexto |

In [ ]:
contexto = '\n\n'.join(
    f"[Fuente {i}] {chunk['source']}: {chunk['text']}"
    for i, (chunk, score) in enumerate(top_3, start=1)
)

# Docs: https://platform.openai.com/docs/guides/text-generation
# API ref: https://platform.openai.com/docs/api-reference/chat
respuesta = client.chat.completions.create(
    model='gpt-4o-mini',
    messages=[
        {'role': 'system', 'content': 'Responde solo usando el contexto.'},
        {'role': 'user', 'content': f'Pregunta: {PREGUNTA}\n\nContexto:\n{contexto}'},
    ],
)

print('Respuesta RAG:', respuesta.choices[0].message.content)

## Cierre

Lo que hicimos manualmente:

1. Dividimos documentos en chunks.
2. Convertimos chunks y pregunta en embeddings.
3. Comparamos vectores con similitud coseno.
4. Elegimos los 3 mejores chunks.
5. Usamos esos chunks como contexto para responder.

Esta es la base conceptual que despues automatizan LangChain y LlamaIndex.